# Nettoyage de la base d'apprentissage

In [31]:
import numpy as np
import pandas as pd
import sys
import os

# On connecte le notebook à tous les fichiers inclus dans le dossier /fonctions
sys.path.append(os.path.abspath("../fonctions"))

%load_ext autoreload
%autoreload 2

from cleaning import *

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [32]:
# On charge la base d'apprentissage et celle des classements FIFA
df = pd.read_csv("../data_finale/base_apprentissage.csv")
df_fifa = pd.read_csv("../data/classement_fifa/fifa_ranking_fin_saison.csv", sep=",", encoding="utf-8-sig")

## Homogénéisation des formats

In [33]:
colonnes_dates = ["date_of_birth", "contract_expiration_date"]

# Application ciblée
df = nettoyer_age_et_dates(
    df, colonnes_dates=colonnes_dates
)

Nettoyage de la colonne 'age'
Colonne 'age' convertie en entiers avec succès.
Traitement des colonnes de dates : ['date_of_birth', 'contract_expiration_date']
Toutes les heures ont été retirées (remises à minuit).
Nettoyage de l'âge et des dates terminé.



## Traitement des doublons

In [34]:
verifier_doublons_metier_et_techniques(df)

Recherche de doublons
 Attention : 787 lignes sont des doublons techniques stricts.
(Même joueur, même saison, même club -> Erreur d'extraction/jointure)

Exemple de lignes techniques concernées :
                player  season         team
28             Willian    2021      Arsenal
29             Willian    2021      Arsenal
30             Willian    2021      Arsenal
37   Emiliano Martínez    2021  Aston Villa
38   Emiliano Martínez    2021  Aston Villa
119           Jorginho    2021      Chelsea

869 lignes correspondent à des doublons de mercato
(Même joueur, même saison, mais clubs différents -> Transferts de mi-saison)

Exemple de joueurs transférés concernés :
                    player  season     team
0   Ainsley Maitland-Niles    2021  Arsenal
14             Joe Willock    2021  Arsenal
16         Martin Ødegaard    2021  Arsenal
17             Mathew Ryan    2021  Arsenal
25          Sead Kolašinac    2021  Arsenal
26        Shkodran Mustafi    2021  Arsenal


{'doublons_techniques': np.int64(787), 'doublons_mercato': np.int64(869)}

In [35]:
df = fusionner_doublons_techniques(df)

Format initial de la base : (17122, 134)
Format après fusion intelligente des doublons : (16335, 134)


In [36]:
df = fusionner_et_recalculer_mercato(df)

Format avant fusion mercato : (16335, 134)
Format après fusion mercato : (15466, 134)
Recalcul des ratios et statistiques par 90 minutes...
Base de données fusionnée et variables recalculées avec exactitude.



c:\Users\LouisHarle\OneDrive - Quadratic\Bureau\Stage_VM\predict_vm_football\fonctions\cleaning.py:252: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  ).agg(aggregation_rules)
c:\Users\LouisHarle\OneDrive - Quadratic\Bureau\Stage_VM\predict_vm_football\fonctions\cleaning.py:252: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  ).agg(aggregation_rules)


## Traitement des variables avec beaucoup de valeurs manquantes

In [37]:
diagnostiquer_valeurs_manquantes(df, seuil=0.01)

Diagnostic des valeurs manquantes (Seuil > 1%)
   • Penalty Kicks_Save% : 94.6% de valeurs manquantes
   • Performance_CS% : 92.8% de valeurs manquantes
   • Performance_Save% : 92.6% de valeurs manquantes
   • Standard_G/SoT : 27.9% de valeurs manquantes
   • contract_expiration_date : 25.2% de valeurs manquantes
   • foot : 17.2% de valeurs manquantes
   • date_of_birth : 17.1% de valeurs manquantes
   • tm_dob_key : 17.1% de valeurs manquantes
   • sub_position : 17.1% de valeurs manquantes
   • tm_join_key_full : 17.1% de valeurs manquantes
   • date : 17.1% de valeurs manquantes
   • market_value_in_eur : 17.1% de valeurs manquantes
   • name : 17.1% de valeurs manquantes
   • tm_join_key : 17.1% de valeurs manquantes
   • valuation_season_year : 17.1% de valeurs manquantes
   • player_id : 17.1% de valeurs manquantes
   • position : 17.1% de valeurs manquantes
   • Standard_SoT% : 16.9% de valeurs manquantes
   • Standard_G/Sh : 16.9% de valeurs manquantes
   • Per 90 Minutes_G+A

In [38]:
# Application de la fonction
df = nettoyer_valeurs_manquantes_ciblees(df)

Début du traitement ciblé des valeurs manquantes...
 -> 11 colonnes de performance nettoyées (NaN -> 0).
 -> Propagation inter-saisons terminée pour 9 colonnes fixes.
Finitions terminées (Derniers NaN résiduels convertis en valeurs neutres).


In [39]:
diagnostiquer_valeurs_manquantes(df, seuil=0.01)

Diagnostic des valeurs manquantes (Seuil > 1%)
   • contract_expiration_date : 25.2% de valeurs manquantes
   • valuation_season_year : 17.1% de valeurs manquantes
   • market_value_in_eur : 17.1% de valeurs manquantes
   • date : 17.1% de valeurs manquantes

Total : 4 colonnes dépassent le seuil de 1%.


## Suppression de colonnes en double

In [40]:
# Exemple de liste de colonnes à jeter après ton analyse de colinéarité
colonnes_redondantes = [
    "Starts_Starts",
    "Standard_PK",
    "Standard_PKatt",
    "Standard_Gls",  # Doublon parfait de Performance_Gls
    "90s",
    "Playing Time_Min%",
    "Performance_SoTA",
    "Team Success_+/-90"
]

# Application
df = supprimer_colonnes_du_dataset(df, colonnes_redondantes)

8 colonne(s) supprimée(s) : ['Starts_Starts', 'Standard_PK', 'Standard_PKatt', 'Standard_Gls', '90s', 'Playing Time_Min%', 'Performance_SoTA', 'Team Success_+/-90']


In [41]:
df

,player,season,team,league,nation,pos,age,born,Playing Time_MP,Playing Time_Starts,...,injury_minor_unknown_nb_d,injury_minor_unknown_nb_m,injury_musculaire,injury_genou,injury_cheville_pied,injury_mollet_tibia,injury_dos_bassin,injury_trauma_severe,injury_medical_repos,injury_minor_unknown
0,Aaron Ciammaglichella,2425,Torino,ITA-Serie A,ITA,MF,19,2005.0,1,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,Aaron Connolly,2021,Brighton,ENG-Premier League,IRL,"FW,MF",20,2000.0,17,9,...,29.0,6.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,Aaron Connolly,2122,Brighton,ENG-Premier League,IRL,FW,21,2000.0,4,1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,Aaron Cresswell,2021,West Ham United,ENG-Premier League,ENG,DF,30,1989.0,36,36,...,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
4,Aaron Cresswell,2122,West Ham United,ENG-Premier League,ENG,DF,31,1989.0,31,31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15461,Šime Vrsaljko,2021,Atlético Madrid,ESP-La Liga,CRO,MF,28,1992.0,9,6,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
15462,Šime Vrsaljko,2122,Atlético Madrid,ESP-La Liga,CRO,"DF,MF",29,1992.0,21,10,...,193.0,10.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0
15463,Ștefan Radu,2021,Lazio,ITA-Serie A,ROU,DF,33,1986.0,31,30,...,24.0,5.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
15464,Ștefan Radu,2122,Lazio,ITA-Serie A,ROU,DF,34,1986.0,10,6,...,36.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0


## Encodage de variables

In [42]:
# Analyser la base des gardiens
var_categorielles = lister_variables_categorielles(df)

Variables catégorielles du dataset :
Liste des variables catégorielles détectées :
   • player (5511 modalités uniques)
   • team (137 modalités uniques)
   • league (5 modalités uniques)
   • nation (129 modalités uniques)
   • pos (10 modalités uniques)
   • join_key (5509 modalités uniques)
   • match_method (17 modalités uniques)
   • date (254 modalités uniques)
   • name (4547 modalités uniques)
   • tm_join_key (4546 modalités uniques)
   • tm_join_key_full (4546 modalités uniques)
   • sub_position (13 modalités uniques)
   • position (5 modalités uniques)
   • foot (3 modalités uniques)

Total : 14 variables catégorielles trouvées.


In [43]:
# Variables catégorielles à traiter
mes_variables = ["pos", "sub_position", "nation", "league", "foot"]

# Lancement de l'encodage
df = encoder_dataset_football(
    df=df,
    colonnes_categoriques=mes_variables,
    df_fifa_historique=df_fifa,
)

Format initial avant encodage : (15466, 126)
Encodage de 'nation' en 10 colonnes binaires Top FIFA (par saison)...
   • Les 10 colonnes classement_FIFA_X ont été injectées.
Profil Joueurs de champ détecté : Encodage Multi-Label de 'pos'.
Encodage One-Hot des colonnes : ['sub_position', 'league', 'foot']
Format final après encodage : (15466, 157)



## Traitement des outliers

In [44]:
(dictionnaire_resultats, liste_lignes_outliers) = detecter_tous_outliers_iqr_trie_filtre(df)

Analyse en cours sur 73 variables numériques continues...
(72 variables binaires exclues)

Variables contenant strictement plus de 5% d'outliers :
Colonne 'height_in_cm' : 3417 outliers détectés (22.09% du dataset)
Colonne 'Starts_Mn/Start' : 2307 outliers détectés (14.92% du dataset)
Colonne 'xg' : 2108 outliers détectés (13.63% du dataset)
Colonne 'np_xg' : 2066 outliers détectés (13.36% du dataset)
Colonne 'xa' : 1959 outliers détectés (12.67% du dataset)
Colonne 'Performance_CrdR' : 1805 outliers détectés (11.67% du dataset)
Colonne 'Performance_Gls' : 1517 outliers détectés (9.81% du dataset)
Colonne 'Team Success_+/-' : 1478 outliers détectés (9.56% du dataset)
Colonne 'Performance_PKatt' : 1474 outliers détectés (9.53% du dataset)
Colonne 'Performance_Crs' : 1464 outliers détectés (9.47% du dataset)
Colonne 'Performance_Off' : 1427 outliers détectés (9.23% du dataset)
Colonne 'Performance_G-PK' : 1307 outliers détectés (8.45% du dataset)
Colonne 'Performance_PK' : 1228 outliers 

In [45]:
df = imputer_donnees_physiques_et_ages(df)

Début du traitement de la taille et du calcul des âges...
 -> Valeurs aberrantes de taille imputées par la médiane du poste.
 -> Colonnes 'born' (année) et 'age' calculées avec succès.
Traitement terminé.


## Traitement du data leakage

In [46]:
df = calculer_jours_contrat_restants(df)

Début du calcul de la durée restante des contrats...
Contrats déjà expirés (valeur négative) : 0
Contrats manquants (NaN)                  : 3895

Statistiques descriptives de la variable calculée :
count    11571.000000
mean      1469.346383
std        721.450833
min          0.000000
25%       1095.000000
50%       1461.000000
75%       1827.000000
max       5113.000000
Calcul terminé avec succès.


In [47]:
# Application de la fonction
df = supprimer_colonnes_inutiles_et_leakage(df)

Début du nettoyage des colonnes avant modélisation...
Identifiants techniques             :  8 colonnes supprimées (join_key, tm_join_key, tm_join_key_full, tm_id, player_id, dob_key, tm_dob_key, match_method)
Doublons d'information              :  4 colonnes supprimées (name, date_of_birth, date, season)
Fuites de données (Data Leakage)    :  2 colonnes supprimées (valuation_season_year, contract_expiration_date)

Nettoyage terminé. Total de colonnes supprimées : 14
Dimensions actuelles du DataFrame : (15466, 144)


In [48]:
df = normaliser_variables_continues(df)

Début du processus de normalisation (MinMax)...
Colonnes binaires (déjà en 0/1, à ne pas normaliser) : 37
Colonnes continues à dupliquer en version '_nor'     : 102


Création des colonnes '_nor' terminée. Total de colonnes actuel : 246


c:\Users\LouisHarle\OneDrive - Quadratic\Bureau\Stage_VM\predict_vm_football\fonctions\cleaning.py:896: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_clean[cols_normalisees_noms] = scaler.fit_transform(
c:\Users\LouisHarle\OneDrive - Quadratic\Bureau\Stage_VM\predict_vm_football\fonctions\cleaning.py:896: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_clean[cols_normalisees_noms] = scaler.fit_transform(
c:\Users\LouisHarle\OneDrive - Quadratic\Bureau\Stage_VM\predict_vm_football\fonctions\cleaning.py:896: PerformanceWarning

## Sauvegarde des bases d'apprentissage mises à jour

In [49]:
df.to_csv(r'..\data_finale\base_apprentissage_maj.csv', index=False, sep=',', encoding='utf-8-sig')

In [50]:
df

,player,team,nation,age,born,Playing Time_MP,Playing Time_Starts,Playing Time_Min,Playing Time_90s,Performance_Gls,...,injury_minor_unknown_nb_m_nor,injury_musculaire_nor,injury_genou_nor,injury_cheville_pied_nor,injury_mollet_tibia_nor,injury_dos_bassin_nor,injury_trauma_severe_nor,injury_medical_repos_nor,injury_minor_unknown_nor,contrat_jours_restants_nor
0,Aaron Ciammaglichella,Torino,ITA,24,2000,1,0,1,0.0,0,...,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.285742
1,Aaron Connolly,Brighton,IRL,20,2000,17,9,791,8.8,2,...,0.107143,0.333333,0.0,0.0,0.0,0.0,0.0,0.0,0.333333,0.214356
2,Aaron Connolly,Brighton,IRL,21,2000,4,1,156,1.7,0,...,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.285742
3,Aaron Cresswell,West Ham United,ENG,31,1989,36,36,3170,35.2,0,...,0.000000,0.333333,0.0,0.5,0.0,0.0,0.0,0.0,0.000000,0.357129
4,Aaron Cresswell,West Ham United,ENG,32,1989,31,31,2726,30.3,2,...,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.285742
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15461,Šime Vrsaljko,Atlético Madrid,CRO,28,1992,9,6,519,5.8,0,...,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.285742
15462,Šime Vrsaljko,Atlético Madrid,CRO,29,1992,21,10,914,10.2,1,...,0.178571,0.000000,0.0,0.0,0.0,0.0,0.5,0.0,0.333333,0.285742
15463,Ștefan Radu,Lazio,ROU,34,1986,31,30,2458,27.3,0,...,0.089286,0.333333,0.0,0.0,0.0,0.0,0.0,0.0,0.333333,0.285742
15464,Ștefan Radu,Lazio,ROU,35,1986,10,6,556,6.2,0,...,0.053571,0.000000,0.0,0.0,0.0,0.0,0.0,0.5,0.333333,0.285742
